Data scraper for San Bernadino, Pomona, humboldt, Somona

Schema:
- Name (First Last)
- School
- Department
- Title
- Major (null if they don't have)

Sources:
San Bernadino: 

- https://www.csusb.edu/campus-directory There is a dropdown to search by department, go through each department option and scrape all of the entries.
- returns a table with firstname, lastname, department, and title

Pomona:

- https://www.cpp.edu/faculty/index-new.shtml (should be 862 entries)
- Each entry has a webpage like https://www.cpp.edu/faculty/{name}/index.shtml which has their title and department in this format Name new line, title newline, department, college

Humboldt:  

- Directory is a search page: https://www.humboldt.edu/its/directory?search=
- results have a table with name, department, title. IF the title field is blank don’t scrape that entry

Somona:

- https://ldaps.sonoma.edu/fasd/ has a drop down and query for each department, returns a table with name, department, and title
- clean title ‘Instr Fac AY’ to Instructional Faculty

## Run the four CSU collectors

Run the setup cell, then each campus cell below. All scraping, parsing, cleaning, and export code is contained in this notebook; no helper script is required.

Outputs are written to `Faculty/Data/`: `csusb_faculty.parquet`, `cpp_faculty.parquet`, `humboldt_faculty.parquet`, and `sonoma_faculty.parquet`.

All files use lowercase columns **name, school, department, title, major**, matching the other scraping notebook. Names are in first-last order. School is the university; department is the academic/administrative unit. Major is null unless explicitly supplied by the directory; it is not inferred from department or biography. Per the source notes, directory staff are retained. Humboldt entries without titles are omitted. Different appointments for the same person are retained; identical rows are deduplicated.

Successful HTTP responses are cached for resumable runs. Set `REFRESH = True` for a fresh scrape. Each campus writes a `*_scrape_report.json` with counts and errors. Failed department/search requests stop that campus export rather than replacing an existing file with incomplete data. Unavailable Pomona profiles retain the feed entry and are flagged in the report.


In [ ]:
# If needed, uncomment and run once in this notebook's kernel:
# %pip install requests beautifulsoup4 pandas pyarrow

import hashlib
import json
import logging
import re
import string
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode, urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# Resolve Faculty/Data whether Jupyter starts in data_scrapers, Faculty,
# CSC4693, or the DataEng workspace.
SCRAPER_DIR = next(
    (candidate for base in (Path.cwd(), *Path.cwd().parents)
     for candidate in (base, base / "data_scrapers", base / "Faculty" / "data_scrapers",
                       base / "CSC4693" / "Faculty" / "data_scrapers")
     if (candidate / "2_csu_data_scraper.ipynb").is_file()),
    None,
)
if SCRAPER_DIR is None:
    raise FileNotFoundError("Open this notebook from within the DataEng project.")
OUTPUT_DIR = SCRAPER_DIR.parent / "Data"
CACHE_DIR = SCRAPER_DIR / ".csu_cache" / "responses"
REFRESH = False  # True ignores cached responses and downloads fresh pages.
REQUEST_DELAY = 0.4
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")


## Shared request, table, and export helpers
The helpers below handle caching, retries, pagination, table headers, department forms, and Parquet export. The campus-specific collectors follow.


In [ ]:
URLS = {
    "csusb": "https://www.csusb.edu/campus-directory",
    "cpp": "https://www.cpp.edu/faculty/index-new.shtml",
    "humboldt": "https://www.humboldt.edu/its/directory?search=",
    "sonoma": "https://ldaps.sonoma.edu/fasd/",
}
SCHOOLS = {
    "csusb": "CSU San Bernardino", "cpp": "Cal Poly Pomona",
    "humboldt": "Cal Poly Humboldt", "sonoma": "Sonoma State University",
}
COLUMNS = ["name", "school", "department", "title", "major"]
LOG = logging.getLogger(__name__)


def clean(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()


def first_last(value):
    """Reorder directory 'Last, First' names, keeping compound surnames."""
    value = clean(value)
    if "," in value:
        last, first = value.split(",", 1)
        value = f"{first.strip()} {last.strip()}"
    return clean(value)


def record(campus, name, department=None, title=None, major=None):
    return dict(zip(COLUMNS, [clean(name), SCHOOLS[campus], clean(department) or None,
                              clean(title) or None, clean(major) or None]))


class Client:
    """Cache successful responses for resumable runs; throttle and retry requests."""

    def __init__(self, cache_dir, delay=0.4, refresh=False):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.delay, self.refresh, self.last_request = max(0, delay), refresh, 0.0
        self.session = requests.Session()
        self.session.headers["User-Agent"] = "CSUFacultyCollector/1.0 (public academic directory research)"
        retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
                      allowed_methods=["GET", "POST"], respect_retry_after_header=True)
        self.session.mount("https://", HTTPAdapter(max_retries=retry))

    def get(self, url, *, data=None):
        method = "POST" if data is not None else "GET"
        key = hashlib.sha256(json.dumps([method, url, data], sort_keys=True).encode()).hexdigest()
        cached = self.cache_dir / f"{key}.html"
        if cached.exists() and not self.refresh:
            return cached.read_text(encoding="utf-8")
        time.sleep(max(0, self.delay - (time.monotonic() - self.last_request)))
        self.last_request = time.monotonic()
        response = self.session.request(method, url, data=data, timeout=(15, 60))
        response.raise_for_status()
        # These sites serve UTF-8 even when their HTTP headers omit charset.
        html = response.content.decode("utf-8", errors="replace")
        cached.write_text(html, encoding="utf-8")
        return html

    def close(self):
        self.session.close()


def soup(html):
    return BeautifulSoup(html, "html.parser")


def table_records(html, campus):
    """Match headers, not column positions (CSUSB has nine columns)."""
    doc = soup(html)
    records, recognized = [], False
    for table in doc.select("table"):
        caption = table.find("caption")
        if campus == "humboldt" and caption and "Personnel Listings" not in caption.get_text():
            continue
        headings = table.select("tr th")
        headers = [clean(h.get_text(" ", strip=True)).lower() for h in headings]
        if not {"department", "title"}.issubset(headers):
            continue
        if "name" not in headers and not {"first name", "last name"}.issubset(headers):
            continue
        recognized = True
        for tr in table.select("tr"):
            cells = tr.find_all("td", recursive=False)
            if not cells:
                continue
            if len(cells) != len(headers):
                raise ValueError(f"{campus}: malformed directory row ({len(cells)} cells, {len(headers)} headers)")
            values = dict(zip(headers, [clean(td.get_text(" ", strip=True)) for td in cells]))
            name = (clean(f"{values['first name']} {values['last name']}") if "first name" in values
                    else first_last(values["name"]))
            title = values["title"]
            if campus == "humboldt" and not title:
                continue
            if campus == "sonoma":
                title = re.sub(r"\bInstr\s+Fac\s+AY\b", "Instructional Faculty", title, flags=re.I)
            if name:
                records.append(record(campus, name, values["department"], title, values.get("major")))
    if not recognized:
        text = doc.get_text(" ", strip=True).lower()
        if not any(message in text for message in ("no results", "no matches", "no people", "no records")):
            raise ValueError(f"{campus}: no recognizable directory table or explicit empty-results message")
    return records


def next_page(html, url):
    doc = soup(html)
    link = doc.select_one('a[rel~="next"], .pager__item--next a')
    if not link:
        return None
    target = urljoin(url, link["href"])
    if urlparse(target).netloc != urlparse(url).netloc:
        raise ValueError(f"Unexpected external pagination link: {target}")
    return target


def read_pages(client, url, campus, *, data=None):
    records, visited = [], set()
    while url:
        if url in visited:
            raise ValueError(f"Pagination loop at {url}")
        visited.add(url)
        html = client.get(url, data=data)
        records.extend(table_records(html, campus))
        url, data = next_page(html, url), None
    return records


def department_options(html, selector):
    select = soup(html).select_one(selector)
    if select is None:
        raise ValueError(f"Department dropdown missing: {selector}")
    options = [(option.get("value", ""), clean(option.get_text())) for option in select.select("option")]
    options = [(value, label) for value, label in options if value and value.lower() != "all"]
    if not options:
        raise ValueError("Department dropdown contains no departments")
    return options


def collect_departments(client, campus, report):
    url = URLS[campus]
    landing = client.get(url)
    field = "field_department_titles" if campus == "csusb" else "dept"
    options = department_options(landing, f'select[name="{field}"]')
    report["departments_total"] = len(options)
    rows = []
    for index, (value, label) in enumerate(options, 1):
        if campus == "csusb":
            target, data = url + "?" + urlencode({field: value}), None
        else:
            form = soup(landing).select_one('select[name="dept"]').find_parent("form")
            target = urljoin(url, form.get("action", ""))
            data = {i["name"]: i.get("value", "") for i in form.select('input[type="hidden"][name]')}
            data[field] = value
        try:
            found = read_pages(client, target, campus, data=data)
            rows.extend(found)
            report["departments"].append({"id": value, "name": label, "rows": len(found)})
        except (requests.RequestException, ValueError) as error:
            report["errors"].append({"url": target, "department": label, "error": str(error)})
            LOG.error("%s / %s: %s", campus, label, error)
        if index % 25 == 0 or index == len(options):
            LOG.info("%s: %d/%d departments, %d rows", campus, index, len(options), len(rows))
    if report["errors"]:
        raise RuntimeError(f"{campus}: {len(report['errors'])} department requests failed; output not replaced")
    return rows


def collect_campus(campus, output_dir=OUTPUT_DIR, *, delay=0.4, refresh=False,
                   cache_dir=CACHE_DIR):
    """Collect and atomically export a campus; also write an audit JSON report.

    Successful cached pages are reused until refresh=True. A failed department
    or search does not replace existing data. Failed Pomona profiles retain the
    feed row and are explicitly listed in the report, with status 'partial'.
    """
    if campus not in URLS:
        raise ValueError(f"Unknown campus: {campus}")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    destination = output_dir / f"{campus}_faculty.parquet"
    report = {"campus": campus, "source_url": URLS[campus], "started_at": datetime.now(timezone.utc).isoformat(),
              "status": "running", "departments": [], "searches": [], "errors": []}
    client = Client(cache_dir, delay, refresh)
    try:
        if campus in {"csusb", "sonoma"}:
            rows = collect_departments(client, campus, report)
        elif campus == "cpp":
            rows = collect_cpp(client, report)
        else:
            rows = collect_humboldt(client, report)
        if not rows:
            raise ValueError(f"{campus}: zero records; refusing to write an empty file")
        frame = pd.DataFrame(rows, columns=COLUMNS).astype("string")
        frame = frame.drop_duplicates().sort_values(["name", "department", "title"], na_position="last").reset_index(drop=True)
        temporary = destination.with_suffix(".parquet.tmp")
        frame.to_parquet(temporary, index=False, engine="pyarrow")
        temporary.replace(destination)
        report.update(status="partial" if report["errors"] else "complete", rows=len(frame),
                      unique_names=frame["name"].nunique(), missing_titles=int(frame["title"].isna().sum()),
                      output=str(destination))
        LOG.info("Wrote %d rows to %s (status: %s)", len(frame), destination, report["status"])
        return frame
    except Exception as error:
        report.update(status="failed", error=str(error))
        raise
    finally:
        client.close()
        report["finished_at"] = datetime.now(timezone.utc).isoformat()
        (output_dir / f"{campus}_scrape_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")



## CSU San Bernardino
Queries every department option from the supplied campus-directory URL and follows every next-page link (the directory displays only 15 entries per page). Combines First Name and Last Name and reads Department and Title by their table headers.


In [ ]:
csusb_df = collect_campus("csusb", OUTPUT_DIR, delay=REQUEST_DELAY, refresh=REFRESH)
print(f"Saved {len(csusb_df):,} rows to {OUTPUT_DIR / 'csusb_faculty.parquet'}")
csusb_df.head()


## Cal Poly Pomona
Discovers the public JSON feed referenced by the supplied directory page, then visits every linked faculty profile for name, title, and department. The feed's `school` field is the department and its `dept` field is the college. The notebook's expected count (862) is a sanity check, not a hard limit; changes in the live count are logged.


In [ ]:
def parse_cpp_profile(html, entry):
    doc = soup(html)
    info = doc.select_one(".fac-info")
    if info is None or info.select_one("h1.profheader") is None:
        raise ValueError("Faculty profile header missing")
    title = info.select_one("p.title")
    affiliation = info.select_one("p.college")
    department = clean(entry.get("school"))  # Feed's 'school' is the department, 'dept' the college.
    if affiliation:
        text = clean(affiliation.get_text(" ", strip=True))
        college = clean(entry.get("dept"))
        if college and text.endswith(college):
            department = text[:-len(college)].rstrip(" ,") or department
        elif re.search(r",\s*(?:College of|[\w .'-]+ College\b)", text):
            department = re.split(r",\s*(?=College of|[\w .'-]+ College\b)", text, maxsplit=1)[0]
        elif text:
            department = text
    major = info.select_one(".major")
    return record("cpp", clean(info.select_one("h1.profheader").get_text(" ", strip=True)),
                  department, title.get_text(" ", strip=True) if title else entry.get("title"),
                  major.get_text(" ", strip=True) if major else None)


def collect_cpp(client, report):
    landing = client.get(URLS["cpp"])
    match = re.search(r'var\s+jsonFeed\s*=\s*[\"\']([^\"\']+)', landing)
    if not match:
        raise ValueError("Pomona directory JSON feed link missing")
    feed_url = urljoin(URLS["cpp"], match.group(1))
    entries = json.loads(client.get(feed_url))["faculty"]
    if not entries:
        raise ValueError("Pomona directory feed is empty")
    report["directory_entries"] = len(entries)
    report["expected_entries_from_notes"] = 862
    if len(entries) != 862:
        LOG.warning("Pomona has %d feed entries; notebook noted 862. Collecting the live count.", len(entries))
    rows = []
    for index, entry in enumerate(entries, 1):
        url = urljoin(URLS["cpp"], entry.get("link", ""))
        try:
            if urlparse(url).hostname not in {"www.cpp.edu", "cpp.edu"} or not entry.get("link"):
                raise ValueError("Missing or external faculty profile link")
            rows.append(parse_cpp_profile(client.get(url), entry))
        except (requests.RequestException, ValueError) as error:
            # Keep the directory-listed person even if their profile is unavailable.
            rows.append(record("cpp", first_last(entry["name"]), entry.get("school"), entry.get("title")))
            report["errors"].append({"url": url, "name": entry["name"], "error": str(error)})
            LOG.warning("Pomona profile unavailable; using feed fields: %s (%s)", url, error)
        if index % 50 == 0 or index == len(entries):
            LOG.info("cpp: %d/%d profiles", index, len(entries))
    return rows



In [ ]:
cpp_df = collect_campus("cpp", OUTPUT_DIR, delay=REQUEST_DELAY, refresh=REFRESH)
print(f"Saved {len(cpp_df):,} rows to {OUTPUT_DIR / 'cpp_faculty.parquet'}")
cpp_df.head()


## Cal Poly Humboldt
An empty search returns instructions rather than people. The live server accepts single-letter searches despite the three-letter hint on the page. Search A?Z, plus the campus email domain and telephone prefix, union the results, and deduplicate. Parse only personnel listings and exclude entries with blank titles. This covers searchable records containing Latin letters or the campus contact identifiers; it does not claim coverage of records containing none of those terms.


In [ ]:
def collect_humboldt(client, report):
    # The live server accepts one-letter searches despite its three-letter UI tip.
    # Searching every letter covers names/titles without requiring an email/phone.
    # Include domain/phone searches as additional coverage for non-Latin names.
    rows = []
    for term in [*string.ascii_lowercase, "humboldt.edu", "826"]:
        url = URLS["humboldt"].split("?", 1)[0] + "?" + urlencode({"search": term})
        found = read_pages(client, url, "humboldt")
        report["searches"].append({"term": term, "rows": len(found)})
        rows.extend(found)
        LOG.info("humboldt: search %r, %d rows with titles", term, len(found))
    return rows



In [ ]:
humboldt_df = collect_campus("humboldt", OUTPUT_DIR, delay=REQUEST_DELAY, refresh=REFRESH)
print(f"Saved {len(humboldt_df):,} rows to {OUTPUT_DIR / 'humboldt_faculty.parquet'}")
humboldt_df.head()


## Sonoma State University
Submits the directory's POST form once for each department dropdown value, including its hidden search fields. Departments with an explicit no-results response are recorded as empty. Replaces `Instr Fac AY` with `Instructional Faculty`.


In [ ]:
sonoma_df = collect_campus("sonoma", OUTPUT_DIR, delay=REQUEST_DELAY, refresh=REFRESH)
print(f"Saved {len(sonoma_df):,} rows to {OUTPUT_DIR / 'sonoma_faculty.parquet'}")
sonoma_df.head()


## Check exports
Review status and profile errors in the per-campus reports before analysis. Counts represent unique name/department/title rows, not necessarily unique people.


In [ ]:
import json
import pandas as pd

summary = []
for campus in ("csusb", "cpp", "humboldt", "sonoma"):
    report = json.loads((OUTPUT_DIR / f"{campus}_scrape_report.json").read_text(encoding="utf-8"))
    summary.append({"campus": campus, "status": report["status"], "rows": report.get("rows"),
                    "unique_names": report.get("unique_names"), "errors": len(report["errors"])})
pd.DataFrame(summary)


## Verified collection: October 4, 2026

| File in `Faculty/Data/` | Distinct rows |
| --- | ---: |
| `csusb_faculty.parquet` | 2,433 |
| `cpp_faculty.parquet` | 861 |
| `humboldt_faculty.parquet` | 1,508 |
| `sonoma_faculty.parquet` | 1,207 |

San Bernardino processed all 240 department options; Sonoma processed all 365. Pomona listed 862 profiles; identical output rows were deduplicated. Twelve Pomona profile URLs returned HTTP 404, so those people were retained from the directory feed with null titles and the URLs were recorded in `cpp_scrape_report.json`. Its status is `partial` because those profile details were unavailable. The other campuses completed successfully. Counts are appointment rows, not unique people.

Validation checked the real site layouts, first-last name ordering, pagination and loop detection, blank-title filtering, Sonoma title normalization, deduplication, preservation of multiple appointments, Parquet round-trip, and protection of existing exports after failed requests.
